In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ========= 一次性构建（已存在则跳过） =========
import os, shutil, random
# 其实下面几个已经不用了，但保留也没关系，不影响运行
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm

output_dir = '/content/drive/MyDrive/EuroSAT_Split_Physical'

if os.path.isdir(os.path.join(output_dir, 'train')) \
   and os.path.isdir(os.path.join(output_dir, 'test')) \
   and os.path.isdir(os.path.join(output_dir, 'valid')):
    print("✅ Found existing dataset splits in Drive. Using them directly.")
else:
    # 不再自动重建，防止误删你自己手动分好的数据
    raise FileNotFoundError(
        f"❌ Did not find train/test/valid under: {output_dir}\n"
        f"请检查：\n"
        f"{output_dir}/train\n{output_dir}/valid\n{output_dir}/test 是否存在。"
    )



In [ ]:
# 检查数据集文件夹信息（不改）
import os
splits = ['train', 'test', 'valid']
for split in splits:
    split_path = os.path.join(output_dir, split)
    print(f"\n数据集划分：{split}")
    for class_name in os.listdir(split_path):
        class_path = os.path.join(split_path, class_name)
        num_images = len(os.listdir(class_path))
        print(f" - 类别 {class_name}: {num_images} 张图片")


In [ ]:
# ========= 统计与可视化（不依赖 load_image）（不改）
import os
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# 统计每类数量
def statistic(train_dir=os.path.join(output_dir, 'train')):
    labels, counts = [], []
    for cname in sorted(os.listdir(train_dir)):
        cpath = os.path.join(train_dir, cname)
        if os.path.isdir(cpath):
            labels.append(cname)
            counts.append(len(os.listdir(cpath)))
    return labels, counts

labels, num_images = statistic()
print(f"📊 Total number of images in dataset: {sum(num_images)}")

# 类别分布柱状图
y_pos = np.arange(len(labels))
plt.figure(figsize=(8, 6))
plt.barh(y_pos, num_images, color='skyblue', edgecolor='black', alpha=0.8)
plt.yticks(y_pos, labels, fontsize=10)
plt.xlabel('Number of Images', fontsize=12)
plt.ylabel('Class Name', fontsize=12)
plt.title('Image Distribution per Class', fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# 简单示例图（每类取1张）
train_dir = os.path.join(output_dir, 'train')
samples, names = [], []
for cname in sorted(os.listdir(train_dir)):
    cpath = os.path.join(train_dir, cname)
    if os.path.isdir(cpath):
        files = os.listdir(cpath)
        if files:
            samples.append(Image.open(os.path.join(cpath, files[0])).convert('RGB'))
            names.append(cname)

n = min(len(samples), 12)
plt.figure(figsize=(12, 8))
for i in range(n):
    plt.subplot(4, 3, i+1)
    plt.imshow(samples[i])
    plt.title(names[i])
    plt.xticks([]); plt.yticks([])
plt.tight_layout(); plt.show()
print("✅ Sample preview done.")


In [ ]:
# ========= 类别名到ID的映射（最小正确写法） =========
def create_label_mapping(train_dir=os.path.join(output_dir, 'train')):
    classes = sorted([
        d for d in os.listdir(train_dir)
        if os.path.isdir(os.path.join(train_dir, d))
    ])
    return {cls: idx for idx, cls in enumerate(classes)}

label2id = create_label_mapping()
print("✅ label2id:", label2id)


In [ ]:
# ====== Cell 1: Setup & utils（不改）======
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
from tqdm import tqdm

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import normalize
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import matplotlib.pyplot as plt
import seaborn as sns

# 设备与随机数
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)
print(f"🚀 Using device: {device}")

# 类别映射（与现有 ./dataset/train 对齐）
def create_label_mapping(train_dir=os.path.join(output_dir, 'train')):
    classes = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
    return {cls: idx for idx, cls in enumerate(classes)}


In [ ]:
# ====== Cell 2: Datasets & loaders（不改）======
def collect_paths_labels(base_dir, label2id):
    """收集 base_dir 下所有图片路径和对应数字标签"""
    img_paths, labels = [], []
    for label, idx in label2id.items():
        class_dir = os.path.join(base_dir, label)
        if not os.path.isdir(class_dir):
            continue
        for fname in os.listdir(class_dir):
            p = os.path.join(class_dir, fname)
            if os.path.isfile(p) and p.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')):
                img_paths.append(p)
                labels.append(idx)
    return img_paths, labels

class ImageFolderPaths(Dataset):
    """从路径读取图片并做预处理"""
    def __init__(self, img_paths, labels, transform):
        self.img_paths = img_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        path = self.img_paths[idx]
        img = Image.open(path).convert('RGB')
        img = self.transform(img)
        label = self.labels[idx]
        return img, label

# 预处理（ResNet 标准）
img_size = 224
transform = T.Compose([
    T.Resize((img_size, img_size)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

# 路径与映射
label2id = create_label_mapping()
id2label = {v: k for k, v in label2id.items()}
class_names = [id2label[i] for i in range(len(id2label))]
num_classes = len(class_names)
print(f"✅ Classes ({num_classes}): {class_names}")

# 收集路径
train_paths, train_labels = collect_paths_labels(os.path.join(output_dir, 'train'), label2id)
valid_paths, valid_labels = collect_paths_labels(os.path.join(output_dir, 'valid'), label2id)
test_paths,  test_labels  = collect_paths_labels(os.path.join(output_dir, 'test'),  label2id)

# DataLoader
batch_size = 64
train_loader = DataLoader(ImageFolderPaths(train_paths, train_labels, transform),
                          batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
valid_loader = DataLoader(ImageFolderPaths(valid_paths, valid_labels, transform),
                          batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(ImageFolderPaths(test_paths,  test_labels,  transform),
                          batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"🔢 Train/Valid/Test sizes: {len(train_paths)} / {len(valid_paths)} / {len(test_paths)}")


In [ ]:
# ====== Cell 3: Pure-ResNet classifier (fine-tuning) ======
import torch
import torch.nn as nn
import torchvision.models as models

# 构建 ResNet18，并替换分类头为 num_classes
backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
in_feats = backbone.fc.in_features
backbone.fc = nn.Linear(in_feats, num_classes)

model = backbone.to(device)

# 损失与优化器
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

# 学习率调度（简易 warmup + cosine）
epochs = 15
warmup_epochs = 1
min_lr = 1e-6
base_lr = 3e-4

def lr_schedule(cur_epoch):
    if cur_epoch < warmup_epochs:
        return (cur_epoch + 1) / warmup_epochs
    t = (cur_epoch - warmup_epochs) / max(1, (epochs - warmup_epochs))
    cosine = 0.5 * (1 + np.cos(np.pi * t))
    return (min_lr / base_lr) + (1 - min_lr / base_lr) * cosine

# 仅用于训练的 DataLoader（保持可比性，前面不改，这里只改 shuffle=True）
from torch.utils.data import DataLoader
train_loader_train = DataLoader(
    ImageFolderPaths(train_paths, train_labels, transform),
    batch_size=64, shuffle=True, num_workers=2, pin_memory=True
)

# 混合精度
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

print("✅ Pure ResNet18 classifier ready (end-to-end fine-tuning).")


In [ ]:
# ====== Cell 4: Train/Validate loops ======
from tqdm import tqdm
import numpy as np

def accuracy_top1(logits, targets):
    pred = logits.argmax(dim=1)
    return (pred == targets).float().mean().item()

@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    total_acc  = 0.0
    total_n    = 0
    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logits = model(images)
            loss = criterion(logits, labels)

        bs = images.size(0)
        total_loss += loss.item() * bs
        total_acc  += accuracy_top1(logits, labels) * bs
        total_n    += bs

    return total_loss / max(1, total_n), total_acc / max(1, total_n)

def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0
    total_acc  = 0.0
    total_n    = 0
    pbar = tqdm(dataloader, desc="Training", leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = images.size(0)
        total_loss += loss.item() * bs
        total_acc  += accuracy_top1(logits, labels) * bs
        total_n    += bs

        pbar.set_postfix(loss=f"{total_loss/max(1,total_n):.4f}",
                         acc=f"{total_acc/max(1,total_n):.4f}")

    return total_loss / max(1, total_n), total_acc / max(1, total_n)


In [ ]:
# ====== Cell 5 (修复版): Fit & save best model by validation accuracy + log history ======
import os

best_val_acc = 0.0
best_ckpt = "./resnet18_pure_best.pt"

# 用于可视化的记录容器
epoch_indices = []
epoch_lrs     = []
tr_losses, tr_accs = [], []
val_losses, val_accs = [], []   # ← 修复：两个空列表

for epoch in range(epochs):
    # 调整学习率
    lr_scale = lr_schedule(epoch)
    for pg in optimizer.param_groups:
        pg["lr"] = base_lr * lr_scale
    cur_lr = optimizer.param_groups[0]['lr']

    # 训练与验证
    print(f"\n[Epoch {epoch+1}/{epochs}] lr={cur_lr:.6f}")
    tr_loss, tr_acc = train_one_epoch(model, train_loader_train, optimizer, device)
    val_loss, val_acc = evaluate(model, valid_loader, device)
    print(f"Train: loss={tr_loss:.4f} acc={tr_acc:.4f} | "
          f"Valid: loss={val_loss:.4f} acc={val_acc:.4f}")

    # 记录
    epoch_indices.append(epoch + 1)
    epoch_lrs.append(cur_lr)
    tr_losses.append(tr_loss); tr_accs.append(tr_acc)
    val_losses.append(val_loss); val_accs.append(val_acc)

    # 保存最佳
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            {"model": model.state_dict(), "num_classes": num_classes},
            best_ckpt
        )
        print(f"💾 Saved best checkpoint -> {best_ckpt} (val_acc={best_val_acc:.4f})")

print(f"\n✅ Training done. Best Val Acc = {best_val_acc:.4f}")



[Epoch 1/15] lr=0.000300


Training:   0%|          | 0/283 [00:00<?, ?it/s]/tmp/ipython-input-1062060593.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
Training:  10%|▉         | 28/283 [03:19<25:02,  5.89s/it, acc=0.8209, loss=0.5500]

In [ ]:
# ====== Cell 5b: Visualization of LR / Loss / Accuracy curves ======
import matplotlib.pyplot as plt

# 学习率曲线
plt.figure(figsize=(6,4))
plt.plot(epoch_indices, epoch_lrs, marker='o', label='Learning Rate')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedule (Warmup + Cosine)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

# 损失曲线（训练/验证）
plt.figure(figsize=(6,4))
plt.plot(epoch_indices, tr_losses, marker='o', label='Train Loss')
plt.plot(epoch_indices, val_losses, marker='o', label='Valid Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

# 准确率曲线（训练/验证）
plt.figure(figsize=(6,4))
plt.plot(epoch_indices, tr_accs, marker='o', label='Train Acc')
plt.plot(epoch_indices, val_accs, marker='o', label='Valid Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ====== Cell 6: Load best & Test evaluation ======
import torch
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 载入最佳权重
ckpt = torch.load("./resnet18_pure_best.pt", map_location=device)
model.load_state_dict(ckpt["model"])
model.eval()

# 推理测试集
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(device, non_blocking=True)
        logits = model(images)
        pred = logits.argmax(dim=1).cpu().numpy()
        all_preds.append(pred)
        all_labels.append(labels.numpy())

Y_pred = np.concatenate(all_preds, axis=0)
Y_test_np = np.concatenate(all_labels, axis=0)

test_acc = accuracy_score(Y_test_np, Y_pred)
print(f"📊 Final Test Accuracy (Pure ResNet18): {test_acc:.4f}\n")

print("📋 Classification Report:")
print(classification_report(Y_test_np, Y_pred, target_names=class_names, digits=4))

# 混淆矩阵（行归一化）
cm = confusion_matrix(Y_test_np, Y_pred, labels=list(range(num_classes)))
cm_norm = cm.astype(np.float64) / (cm.sum(axis=1, keepdims=True) + 1e-12)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt='.2f',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Normalized Confusion Matrix (Pure ResNet18 Finetune)', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# ====== Cell 7: 可视化模型在测试集上的预测结果 ======
import random
import matplotlib.pyplot as plt
from PIL import Image

# 确保模型是 eval 模式
model.eval()

# 随机选取若干测试样本
n_show = 12
indices = random.sample(range(len(test_paths)), n_show)

plt.figure(figsize=(14, 9))
for i, idx in enumerate(indices):
    path = test_paths[idx]
    true_label = id2label[test_labels[idx]]

    # 读取图像并预测
    img = Image.open(path).convert('RGB')
    x = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
        pred_id = logits.argmax(dim=1).item()
    pred_label = id2label[pred_id]

    # 显示
    plt.subplot(3, 4, i + 1)
    plt.imshow(img)
    color = 'green' if pred_label == true_label else 'red'
    plt.title(f"Pred: {pred_label}\nTrue: {true_label}", color=color, fontsize=10)
    plt.axis('off')

plt.suptitle("Sample Predictions on Test Images (✅=Correct, ❌=Wrong)", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


In [ ]:
# ====== Cell 7: 读取一整张 Sentinel-2 原图 ======
from PIL import Image
import numpy as np

# TODO: 换成你自己的图片路径
raw_image_path = "/content/2025-08-18-00_00_2025-08-18-23_59_Sentinel-2_L2A_True_Color.png"   # 比如: "/content/xxx.png" 或 .tif

big_img = Image.open(raw_image_path).convert("RGB")
W, H = big_img.size
print(f"原图大小: {W} x {H}")

# 预览一下（缩小显示）
display(big_img.resize((min(W, 800), int(H * min(W, 800) / W))))




In [ ]:
# ====== Cell 8: 把整图切成 64x64 小块 ======

patch_size = 64   # EuroSAT patch 大小
stride = patch_size

# 只使用能整除的一部分区域
n_cols = W // patch_size          # 列数
n_rows = H // patch_size          # 行数

crop_w = n_cols * patch_size      # 实际用于分类的宽度
crop_h = n_rows * patch_size      # 实际用于分类的高度

print(f"用于网格的区域大小: {crop_w} x {crop_h}")
print(f"patch 网格: {n_rows} 行 x {n_cols} 列，共 {n_rows * n_cols} 个 patch")

# 先裁剪到可整除区域
big_img_cropped = big_img.crop((0, 0, crop_w, crop_h))

patches = []
for r in range(n_rows):
    for c in range(n_cols):
        left   = c * stride
        upper  = r * stride
        right  = left + patch_size
        lower  = upper + patch_size
        patch  = big_img_cropped.crop((left, upper, right, lower))
        patches.append(patch)

print("✅ patch 提取完成")


In [ ]:
# ====== Cell 9: 用纯 ResNet18 对所有 patch 做预测 ======
import torch

model.eval()
batch_size_eval = 128

all_preds = []

with torch.no_grad():
    for i in range(0, len(patches), batch_size_eval):
        batch_patches = patches[i : i + batch_size_eval]

        # 使用前面定义好的 transform（Resize 到 224 + Normalize）
        batch_tensors = [transform(p) for p in batch_patches]
        batch = torch.stack(batch_tensors).to(device)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logits = model(batch)
            preds = logits.argmax(dim=1).cpu().numpy()

        all_preds.extend(preds)

all_preds = np.array(all_preds)
print("✅ 预测完成，预测数量：", len(all_preds))

# reshape 成 (n_rows, n_cols) 网格
pred_grid = all_preds.reshape(n_rows, n_cols)
pred_grid.shape


In [ ]:
# ====== Final Visualization: 输出整图尺寸 + 裁剪尺寸 + 显示预测结果 ======
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# 1) 扩展 patch 预测到像素级别
patch_size = 64
pred_full_res = np.kron(pred_grid, np.ones((patch_size, patch_size), dtype=int))

print("pred_full_res shape:", pred_full_res.shape)  # (crop_h, crop_w)

# 2) 准备离散 colormap
color_list = [
    "#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00",
    "#ffff33", "#a65628", "#f781bf", "#999999", "#66c2a5",
    "#fc8d62", "#8da0cb", "#e78ac3", "#a6d854", "#ffd92f",
    "#e5c494", "#b3b3b3"
]
assert len(color_list) >= len(class_names)
cmap = ListedColormap(color_list[:len(class_names)])

# 3) 可视化图形
plt.figure(figsize=(15, 7))

# 左图：原图裁剪部分
plt.subplot(1, 2, 1)
plt.imshow(big_img_cropped)
plt.title(
    f"Original (Full {W}×{H} → Cropped {crop_w}×{crop_h})",
    fontsize=12
)
plt.axis("off")

# 右图：预测图
plt.subplot(1, 2, 2)
im = plt.imshow(pred_full_res, cmap=cmap,
                vmin=0, vmax=len(class_names)-1,
                interpolation="nearest")
plt.title("Prediction (Pure ResNet18, Pixel-level Expansion)", fontsize=12)
plt.axis("off")

# 颜色条
cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
cbar.set_ticks(range(len(class_names)))
cbar.set_ticklabels(class_names)

plt.tight_layout()
plt.show()

# 4) 保存图像文件
plt.imsave("prediction_cropped_resnet.png", pred_full_res, cmap=cmap,
           vmin=0, vmax=len(class_names)-1)
big_img_cropped.save("original_cropped_resnet.png")

print("📝 原图尺寸：", (W, H))
print("✂️ 裁剪后尺寸：", (crop_w, crop_h))
print("✅ 已保存：prediction_cropped_resnet.png  和  original_cropped_resnet.png")
